# Training Demo

This notebook prepares a tiny demo dataset from `CT_RATE_demo_data`, writes a short-run config, and runs only a few training steps. A GPU is strongly recommended.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DEMO_ROOT = PROJECT_ROOT / 'CT_RATE_demo_data'
DEMO_DIR = PROJECT_ROOT / 'demo'
TRAINING_DATA_DIR = DEMO_DIR / 'training_demo_npz'
TRAINING_DATA_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = DEMO_DIR / 'training_demo_config.yaml'
OUTPUT_DIR = DEMO_DIR / 'training_demo_runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Demo NPZ dir: {TRAINING_DATA_DIR}')
print(f'Demo config: {CONFIG_PATH}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
command_template = [
    sys.executable,
    str(PROJECT_ROOT / 'training_file_generation' / 'batch_training_object_generation.py'),
    '--single_case',
    '',
    '--output_dir',
    str(TRAINING_DATA_DIR),
]

for projection in ['PA', 'LR']:
    case_dir = DEMO_ROOT / f'CT_RATE_projections_{projection}' / 'train_1_a_2'
    command = command_template.copy()
    command[3] = str(case_dir)
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)

print('Generated NPZ files:', sorted(path.name for path in TRAINING_DATA_DIR.glob('*.npz')))

In [3]:
demo_config = {
    'model': {
        'model_size': 3,
        'input_image_size': 224,
    },
    'data': {
        'training_data_path': str(TRAINING_DATA_DIR),
        'train_val_split': 0.5,
        'enable_augmentation': False,
        'max_structures_for_multistructure_tasks': 2,
        'available_tasks': ['detection', 'orientation_identification'],
    },
    'training': {
        'num_train_epochs': 1,
        'max_steps': 3,
        'per_device_train_batch_size': 1,
        'per_device_eval_batch_size': 1,
        'gradient_accumulation_steps': 1,
        'auto_find_batch_size': False,
        'learning_rate': 5.0e-5,
        'weight_decay': 0.01,
        'adam_beta2': 0.999,
        'warmup_steps': 0,
    },
    'logging': {
        'logging_steps': 1,
        'save_total_limit': 1,
        'run_name': 'chexanatomy_demo_training',
    },
    'optimization': {
        'use_lora': True,
        'use_qlora': False,
        'freeze_vision_tower': True,
        'freeze_mm_projector': False,
        'lora_r': 8,
        'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    },
    'paths': {
        'model_output_dir': str(OUTPUT_DIR),
        'vqvae_model_path': './models/vae-oid.npz',
    },
    'wandb': {
        'project': '',
        'entity': '',
        'tags': ['demo'],
        'notes': 'Short demo run',
        'log_model': False,
        'log_freq': 1,
        'save_code': False,
    },
}

with CONFIG_PATH.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(demo_config, handle, sort_keys=False)

print(CONFIG_PATH.read_text())

model:
  model_size: 3
  input_image_size: 224
data:
  training_data_path: /home/sgatidis/Projects/cheXanatomy/demo/training_demo_npz
  train_val_split: 0.5
  enable_augmentation: false
  max_structures_for_multistructure_tasks: 2
  available_tasks:
  - detection
  - orientation_identification
training:
  num_train_epochs: 1
  max_steps: 3
  per_device_train_batch_size: 1
  per_device_eval_batch_size: 1
  gradient_accumulation_steps: 1
  auto_find_batch_size: false
  learning_rate: 5.0e-05
  weight_decay: 0.01
  adam_beta2: 0.999
  warmup_steps: 0
logging:
  logging_steps: 1
  save_total_limit: 1
  run_name: chexanatomy_demo_training
optimization:
  use_lora: true
  use_qlora: false
  freeze_vision_tower: true
  freeze_mm_projector: false
  lora_r: 8
  lora_target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
paths:
  model_output_dir: /home/sgatidis/Projects/cheXanatomy/demo/training_demo_runs
  vqvae_model_path: ./models/vae-oid.npz
wandb:
  project: ''
  entity: ''
  tags:
  

## Run a few training steps

The next cell runs the real training script with `max_steps: 3`. Depending on your hardware, model download and initialization can still take a while.

In [ ]:
def gpu_memory_summary():
    try:
        result = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=index,name,memory.total,memory.used,memory.free',
                '--format=csv,noheader,nounits',
            ],
            check=True,
            text=True,
            capture_output=True,
        )
    except Exception as exc:
        print(f'Could not query nvidia-smi: {exc}')
        return []

    rows = []
    for line in result.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(',')]
        if len(parts) != 5:
            continue
        idx, name, total, used, free = parts
        rows.append({
            'index': int(idx),
            'name': name,
            'total_mb': int(total),
            'used_mb': int(used),
            'free_mb': int(free),
        })
    return rows


gpu_rows = gpu_memory_summary()
if gpu_rows:
    print('GPU memory status:')
    for row in gpu_rows:
        print(
            f"  GPU {row['index']}: {row['name']} | "
            f"free={row['free_mb']} MiB / total={row['total_mb']} MiB"
        )

    max_free_mb = max(row['free_mb'] for row in gpu_rows)
    if max_free_mb < 8192:
        raise RuntimeError(
            'Not enough free GPU memory for the training demo. '
            'Free other GPU jobs and rerun this cell.'
        )

training_command = [
    sys.executable,
    '-u',
    str(PROJECT_ROOT / 'training' / 'train_paligemma.py'),
    '--config',
    str(CONFIG_PATH),
]

env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Running:', ' '.join(training_command))
result = subprocess.run(
    training_command,
    cwd=PROJECT_ROOT,
    env=env,
    text=True,
    capture_output=True,
    check=False,
    )

print('Return code:', result.returncode)
if result.stdout:
    print('STDOUT:\n' + result.stdout)
if result.stderr:
    print('STDERR:\n' + result.stderr)

if result.returncode != 0:
    raise RuntimeError('Training demo failed; see stdout/stderr above.')

In [ ]:
for path in sorted(OUTPUT_DIR.rglob('*')):
    print(path.relative_to(PROJECT_ROOT))